# RSNA Knee Abnormality Detection - Training (Study-Level with MedDINOv3)

Processes all series for a study in a single forward pass.

In [2]:
# !pip install pydicom timm albumentations scikit-learn tqdm

import os, math, random, warnings
import cv2
import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import timm
from tqdm import tqdm
import multiprocessing as mp
try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

from model_rad_dino_simplified import KneeAbnormalityModelWithRadDino, pad_image

warnings.filterwarnings('ignore')
print('✓ All imports successful')

KeyboardInterrupt: 

In [ ]:
class Config:
    KAGGLE_DIR    = Path('/run/media/mahyar/Samsung-990-PRO/Projects/Kaggle/rsna-knee-abnormality-detection')
    PYTHON_DIR    = Path('/run/media/mahyar/Samsung-990-PRO/Projects/Python/RSNA Knee Abnormality Detection')
    TRAIN_CSV     = PYTHON_DIR / 'train_filled_corrected.csv'
    SERIES_CSV    = PYTHON_DIR / 'train_series.csv'
    TRAIN_IMG_DIR = KAGGLE_DIR / 'train_series'
    SAVE_PATH     = str(PYTHON_DIR / 'best_model_meddinov3.pth')

    # ── Image processing ──────────────────────────────────────────────
    MAX_SLICES_PER_SERIES = 12
    LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus',
              'Medial OA','Lateral OA','PF OA','Effusion',
              'Synovitis',"Baker's",'Contusion','Fracture']
    NUM_LABELS = 12

    PLANE_TO_IDX = {'Sagittal': 0, 'Axial': 1, 'Coronal': 2}
    NUM_PLANES   = 3

    # ── Model dimensions ──────────────────────────────────────────────
    SLICE_DIM    = 512
    META_DIM     = 256
    SLICE_ENC_HEADS  = 8
    SLICE_ENC_LAYERS = 2
    DROPOUT      = 0.3

    # ── Train / val split ─────────────────────────────────────────────
    VAL_FRACTION = 0.20   # 20% val, 80% train

    # ── Training hyperparameters ──────────────────────────────────────
    BATCH_SIZE   = 4
    NUM_WORKERS  = 4
    EPOCHS       = 30
    LR           = 1e-4
    MIN_LR       = 1e-6
    WEIGHT_DECAY = 1e-4
    USE_AMP      = True

    DEBUG_SUBSET = None # Set to e.g. 100 for fast debugging
    SEED         = 42
    DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = Config()
print(f"Device : {cfg.DEVICE}")

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)

In [ ]:
# ── Training / Validation Dataset ─────────────────────────────────────────────
class KneeStudyTrainDataset(Dataset):
    '''
    Groups all series for a study together for training.
    Returns slices, metadata, and 12-class target labels.
    '''
    def __init__(self, df_studies: pd.DataFrame, df_series: pd.DataFrame, img_dir: Path, cfg, is_train: bool = True):
        self.cfg = cfg
        self.img_dir = img_dir
        self.is_train = is_train
        
        # Group series by study_id
        self.studies = []
        for _, study_row in df_studies.iterrows():
            study_id = str(study_row['StudyInstanceUID'])
            
            # Get series for this study
            study_series = df_series[df_series['StudyInstanceUID'] == study_id]
            if len(study_series) == 0:
                continue
                
            series_list = []
            for _, row in study_series.iterrows():
                series_list.append({
                    'series_id': str(row['SeriesInstanceUID']),
                    'plane_id': int(cfg.PLANE_TO_IDX.get(str(row['Anatomical_Plane']), 0)),
                    'fluid_sensitive': int(row.get('Fluid_Sensitive', 0)),
                    'fat_suppression': int(row.get('Fat_Suppression', 0))
                })
                
            # Extract labels (nan to 0 if any missing)
            labels = study_row[cfg.LABELS].fillna(0).values.astype(np.float32)
            
            self.studies.append({
                'study_id': study_id,
                'series': series_list,
                'labels': labels
            })
            
        print(f"  {'Train' if is_train else 'Val'}: {len(self.studies)} studies")

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, idx):
        study = self.studies[idx]
        study_id = study['study_id']
        labels = study['labels']
        
        all_slices = []
        all_plane_ids = []
        all_fluids = []
        all_fats = []
        
        for sinfo in study['series']:
            path = self.img_dir / study_id / sinfo['series_id']
            if not path.exists():
                continue
            dcm_files = sorted(path.glob('*.dcm'))[:self.cfg.MAX_SLICES_PER_SERIES]
            for dcm_file in dcm_files:
                dcm = pydicom.dcmread(str(dcm_file))
                img = dcm.pixel_array.astype(np.float32)
                img = (img - img.min()) / (img.max() - img.min() + 1e-6)
                
                # Very simple augmentations for training
                if self.is_train and random.random() > 0.5:
                    img = np.fliplr(img).copy()
                    
                img = cv2.resize(img, (224, 224)) # Base size, will be padded natively
                img = np.stack([img, img, img], axis=0) # (3, H, W)
                
                all_slices.append(torch.from_numpy(img))
                all_plane_ids.append(sinfo['plane_id'])
                all_fluids.append(sinfo['fluid_sensitive'])
                all_fats.append(sinfo['fat_suppression'])
                
        if len(all_slices) == 0:
            # Fallback if no images found
            all_slices.append(torch.zeros(3, 224, 224))
            all_plane_ids.append(0)
            all_fluids.append(0)
            all_fats.append(0)
            
        return {
            'study_id': study_id,
            'slices': torch.stack(all_slices), # (S, 3, H, W)
            'plane_ids': torch.tensor(all_plane_ids, dtype=torch.long),
            'fluid_sensitive': torch.tensor(all_fluids, dtype=torch.long),
            'fat_suppression': torch.tensor(all_fats, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.float32)
        }

In [ ]:
# ── Collate function ──────────────────────────────────────────────────────────
def collate_fn(batch: List[Dict]) -> Dict:
    max_s = max(len(b['slices']) for b in batch)
    
    padded_slices = []
    padded_planes = []
    padded_fluids = []
    padded_fats = []
    slice_masks = []
    study_ids = []
    labels = []
    
    for b in batch:
        s = len(b['slices'])
        study_ids.append(b['study_id'])
        labels.append(b['labels'])
        
        if s < max_s:
            pad_img = torch.zeros(max_s - s, *b['slices'].shape[1:])
            padded_slices.append(torch.cat([b['slices'], pad_img], dim=0))
            
            pad_idx = torch.zeros(max_s - s, dtype=torch.long)
            padded_planes.append(torch.cat([b['plane_ids'], pad_idx], dim=0))
            padded_fluids.append(torch.cat([b['fluid_sensitive'], pad_idx], dim=0))
            padded_fats.append(torch.cat([b['fat_suppression'], pad_idx], dim=0))
            
            mask = torch.cat([torch.zeros(s, dtype=torch.bool), torch.ones(max_s - s, dtype=torch.bool)])
            slice_masks.append(mask)
        else:
            padded_slices.append(b['slices'])
            padded_planes.append(b['plane_ids'])
            padded_fluids.append(b['fluid_sensitive'])
            padded_fats.append(b['fat_suppression'])
            slice_masks.append(torch.zeros(s, dtype=torch.bool))
            
    images = torch.stack(padded_slices) # (B, max_s, 3, 224, 224)
    # Pad natively for MedDINOv3 to 14x multiples
    images = pad_image(images, patch_size=14) # (B, max_s, 3, H, W)
    
    return {
        'study_ids': study_ids,
        'images': images,
        'slice_mask': torch.stack(slice_masks),
        'plane_ids': torch.stack(padded_planes),
        'fluid_sensitive': torch.stack(padded_fluids),
        'fat_suppression': torch.stack(padded_fats),
        'labels': torch.stack(labels),
    }

In [ ]:
# ── Loss ──────────────────────────────────────────────────────────────────────
class WeightedBCELoss(nn.Module):
    def __init__(self, pos_weight=None):
        super().__init__()
        self.pos_weight = pos_weight

    def forward(self, pred, target):
        if self.pos_weight is not None:
            return F.binary_cross_entropy_with_logits(pred, target, pos_weight=self.pos_weight)
        return F.binary_cross_entropy_with_logits(pred, target)

def compute_class_weights(df, cfg):
    weights = []
    for label in cfg.LABELS:
        pos = (df[label] == 1).sum()
        neg = (df[label] == 0).sum()
        weights.append(neg / (pos + 1e-6))
    return torch.tensor(weights, dtype=torch.float32)

In [ ]:
# ── Train & Eval Epochs ───────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, scaler, cfg):
    model.train()
    losses = []
    
    pbar = tqdm(loader, desc='Train')
    for batch in pbar:
        images = batch['images'].to(cfg.DEVICE)
        slice_mask = batch['slice_mask'].to(cfg.DEVICE)
        plane_ids = batch['plane_ids'].to(cfg.DEVICE)
        fluid_sensitive = batch['fluid_sensitive'].to(cfg.DEVICE)
        fat_suppression = batch['fat_suppression'].to(cfg.DEVICE)
        labels = batch['labels'].to(cfg.DEVICE)
        
        optimizer.zero_grad()
        
        if cfg.USE_AMP:
            with autocast():
                logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
        losses.append(loss.item())
        pbar.set_postfix({'loss': {np.mean(losses):.4}})
        
    return np.mean(losses)

@torch.no_grad()
def eval_epoch(model, loader, criterion, cfg):
    model.eval()
    losses = []
    all_preds, all_labels = [], []
    
    for batch in tqdm(loader, desc='Val'):
        images = batch['images'].to(cfg.DEVICE)
        slice_mask = batch['slice_mask'].to(cfg.DEVICE)
        plane_ids = batch['plane_ids'].to(cfg.DEVICE)
        fluid_sensitive = batch['fluid_sensitive'].to(cfg.DEVICE)
        fat_suppression = batch['fat_suppression'].to(cfg.DEVICE)
        labels = batch['labels'].to(cfg.DEVICE)
        
        if cfg.USE_AMP:
            with autocast():
                logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
                loss = criterion(logits, labels)
        else:
            logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
            loss = criterion(logits, labels)
            
        losses.append(loss.item())
        preds = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.cpu().numpy())
        
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    val_loss = np.mean(losses)
    try:
        val_auc = roc_auc_score(all_labels, all_preds, average='macro')
    except ValueError:
        val_auc = 0.5
        
    return val_loss, val_auc

In [1]:
def main():
    print('Loading metadata...')
    df_train = pd.read_csv(cfg.TRAIN_CSV)
    df_series = pd.read_csv(cfg.SERIES_CSV)
    
    if cfg.DEBUG_SUBSET:
        df_train = df_train.head(cfg.DEBUG_SUBSET)
        
    pos_weights = compute_class_weights(df_train, cfg).to(cfg.DEVICE)
    
    # Split by study
    train_studies, val_studies = train_test_split(
        df_train, test_size=cfg.VAL_FRACTION, random_state=cfg.SEED
    )
    
    ds_train = KneeStudyTrainDataset(train_studies, df_series, cfg.TRAIN_IMG_DIR, cfg, is_train=True)
    ds_val = KneeStudyTrainDataset(val_studies, df_series, cfg.TRAIN_IMG_DIR, cfg, is_train=False)
    
    ld_train = DataLoader(ds_train, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn)
    ld_val = DataLoader(ds_val, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True,
                        collate_fn=collate_fn)
                        
    print('\nInitializing model...')
    model = KneeAbnormalityModelWithRadDino(cfg).to(cfg.DEVICE)
    
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable_params:,} / Total: {total_params:,}  ({100*trainable_params/total_params:.1f}%)')
    print('MedDINOv3 backbone frozen ✓')
    
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=cfg.MIN_LR)
    criterion = WeightedBCELoss(pos_weight=pos_weights)
    scaler = GradScaler() if cfg.USE_AMP else None
    
    best_auc = 0
    for epoch in range(cfg.EPOCHS):
        print(f'\n--- Epoch {epoch+1}/{cfg.EPOCHS} ---')
        
        train_loss = train_epoch(model, ld_train, optimizer, criterion, scaler, cfg)
        val_loss, val_auc = eval_epoch(model, ld_val, criterion, cfg)
        scheduler.step()
        
        print(f'Train loss: {train_loss:.4f}  Val loss: {val_loss:.4f}  Val AUC: {val_auc:.4f}')
        
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_auc': val_auc
            }, cfg.SAVE_PATH)
            print(f'✓ Checkpoint saved (AUC: {val_auc:.4f})')
            
if __name__ == '__main__':
    main()
    pass

Loading metadata...


Traceback (most recent call last):
  File "/home/mahyar/.local/share/JetBrains/Toolbox/apps/pycharm/plugins/python-ce/helpers/debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_frame.py", line 1289, in handle_exception
    py_db.do_wait_suspend(thread, frame, "exception", arg, exception_type=exception_type)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mahyar/.local/share/JetBrains/Toolbox/apps/pycharm/plugins/python-ce/helpers/debugpy/_vendored/pydevd/pydevd.py", line 2234, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, trace_suspend_type, from_this_thread, frames_tracker)
  File "/home/mahyar/.local/share/JetBrains/Toolbox/apps/pycharm/plugins/python-ce/helpers/debugpy/_vendored/pydevd/pydevd.py", line 2303, in _do_wait_suspend
    notify_event.wait(wait_timeout)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/usr/lib64/python3.14/threading.py", line 670, in wait
    signaled = self._cond.wai

NameError: name 'pd' is not defined